<h1>How to Extract <i><u>S</u>tack <u>O</u>verflow <u>R</u>ecommendations <u>D</u>ataset (SORD)</i> from Stack Overflow Data Dump</h1>

This notebook provides step by step guideline to extract recommendation related questions, answers and comments from Stack Overflow data dump. Before proceeding further, we assume that the readers have already followed instruction given in the <u>StackOverflow_DataDump_Parsers</u> notebook and have converted the XML files included in data dump to SQL Server database tables, particularly <u>Posts.xml</u> and <u>Comments.xml</u> to <u>StackOverflowPosts</u> (table located in <u>StackOverflowPostsDb_Oct2025</u>) and <u>StackOverflowComments</u> (table located in <u>StackOverflowCommentsDb_Oct2025 database</u>) respectively.

<h2>1. Extracting Questions and Answers from Posts</h2>

We used the following code snippet to extract questions and answers from Posts.

<h2>2. Enable Full-text Search in MS SQL Server</h2>

To enable full text search we followed the instructions given <a href='https://www.mssqltips.com/sqlservertip/6841/add-full-text-search-sql-server/'>here</a>. For the sake of completeness, the main steps are given below (assuming that your SQL Server installation includes full text search feature enabled). 

<h4>2.1 To Check Whether Full Text Search is enabled on relevant databases</h4>

If it is not already enabled, enable it using following command (only once per DB),

<h4>2.2 Create Full-Text Catalog in EACH DATABASE</h4>

<h4>2.3 Check which indexes your table already has</h4>

You need to have a user created index to be able to successfully perform this step. <u>StackOverflowPosts</u> will have it based on how we created it (refer to  <u>CONSTRAINT PK_AutoIdPK PRIMARY KEY CLUSTERED (AutoIdPK)</u>) but since we have created the <u>StackOverflowQuestions</u> and <u>StackOverflowAnswers</u> using Select Into it only copies data without any indexes or constraints. To create the corresponding indexes in <u>StackOverflowQuestions</u> and <u>StackOverflowAnswers</u>, we need to alter the tables as follows: 

<h4>2.4 Create Full-Text Indexes on Each Table</h4>


	if you get an error such as 
	Msg 7609, Level 17, State 5, Line 3
	Full-Text Search is not installed, or a full-text component cannot be loaded
	Go to  C:\Program Files\Microsoft SQL Server\160\Setup Bootstrap\SQL2022\setup.exe and launch the SQL Server Installation Center. 
	Select Installation from the left menu and choose New SQL Server standalone installation or add features to an existing installation which will ask you to browse for sql server 2022 installation media. Select C:\SQL2022\Developer_ENU if you are using Developer Edition of SQL Server. This will open another SQL Server 2022 Setup wizard. From this wizard choose an existing installation and add Full-text and Semantic Extractions for Search feature.    

  Now that the full-text search is enabled, you can execute the above query to create full-text indexes. But now it may give an error like the following: 
  'AutoIdPK' is not a valid index to enforce a full-text search key. A full-text search key must be a unique, non-nullable, single-column index which is not offline, is not defined on a non-deterministic or imprecise nonpersisted computed column, does not have a filter, and has maximum size of 900 bytes. Choose another index for the full-text key.
  This happens because while creating the Table StackOverflowPosts (and StackOverflowQuestions/StackOverflowAnswers) we only created column AutoIdPK but not the index, so we now need to alter the table for creating the index as well. 

<h2>3. Filter Records based on Set of Keywords</h2>

Create a new database named <u>SORecommendationsDb</u> which will now contain the filtered records.

<h3>3.1 Exact Matching Using Full Text Search feature and CONTAIN keyword</h3>

<h4>3.1.1 Filter <i><u>Questions</u></i> that contain the keywords in <i><u>Title</u></i></h4>

<h4>3.1.2 Filter <i><u>Questions</u></i> that contain the keywords in <i><u>Body</u></i></h4>

<h4>3.1.3 Filter <i><u>Answers</u></i> that contain the keywords</h4>

<h4>3.1.4 Filter <i><u>Comments</u></i> that contain the keywords </h4>

<h3>3.2 Substring Matching Using Wildcard Search with LIKE keyword</h3>

<h4>3.2.1 Filter <i><u>Questions</u></i> that contain the keywords in <i><u>Title</u></i></h4>

<h4>3.2.2 Filter <i><u>Questions</u></i> that contain the keywords in <i><u>Body</u></i></h4>

<h4>3.2.3 Filter <i><u>Answers</u></i> that contain the keywords </h4>

<h4>3.2.4 Filter <i><u>Comments</u></i> that contain the keywords </h4>

<h3>3.3 Extracting LIKE - CONTAIN records which may have false positives / negative results or can be still useful in some aspect</h3>

<h4>3.3.1 Questions Title</h4>

<h4>3.3.2 Questions Body</h4>

<h4>3.3.3 Answers</h4>

<h4>3.3.4 Comments</h4>

<h2>4. Export Results to CSV</h2>

We observed that some columns particularly in case of questions and answers were not providing any additional information for any record. This happened because both questions and answers are actually placed in the same table i.e. posts. So some columns are only applicable for questions and vice versa. Hence, before exporting the data to csv files we excluded such columns. The following query was used to compare the min and max values of each column in the respective tables. If the min and max value for a column is same, it means that the column contains same value for all rows and does not provide any important information. 

Similar to the above SQL query, we checked all the tables for question title, question body, answers and comments. We have excluded the following columns in case of questions.
<ol>
<li>PostTypeId (1 for all questions)</li>
<li>ParentId (0 for all questions)</li>
<li>DeletionDate (NULL for all questions)</li>
<li>FavouriteCount (NULL for all questions)</li>
</ol>
Following columns were excluded in case of answers.
<ol>
<li>PostTypeId (2 for all answers)</li>
<li>AcceptedAnswerId (0 for all answers)</li>
<li>DeletionDate (NULL for all answers)</li>
<li>DeletionDate (NULL for all answers)</li>
<li>ViewCount (0 for all answers)</li>
<li>Title (Empty string for all answers)</li>
<li>Tags (Empty string for all answers)</li>
<li>AnswerCount (0 for all answers)</li>
<li>FavouriteCount (NULL for all answers)</li>
</ol>

<h3>4.1 Retrieve Only Relevant Columns</h3>

We used the following SQL queries to get all records (in case of contains tables). You can use the similar pattern in case of likeminuscontain tables as well. We used the <i>Save Results As</i> option in SQL Server Management Studio to save the results in the respective CSV files.